# 🛒 Olist E-Commerce — Exploratory Data Analysis

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Scope:** ~99,000 orders placed between 2016–2018 across Brazil

## Objectives
1. Understand overall revenue and top-performing product categories
2. Analyze delivery performance and its impact on customer satisfaction
3. Identify geographic patterns in delivery delays
4. Profile high-value customers and repeat purchase behaviour
5. Examine payment method distribution and seller performance
6. Visualise monthly revenue trends

---

## 1. Setup — Imports & Style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')          # suppress FutureWarning / pandas noise
sns.set(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

print('Libraries loaded successfully.')

## 2. Load Data

The Olist dataset consists of **8 relational CSV files**. We load each one and immediately inspect its shape so we know what we're working with before merging.

In [ ]:
orders      = pd.read_csv('olist_orders_dataset.csv')
customers   = pd.read_csv('olist_customers_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
products    = pd.read_csv('olist_products_dataset.csv')
payments    = pd.read_csv('olist_order_payments_dataset.csv')
reviews     = pd.read_csv('olist_order_reviews_dataset.csv')
sellers     = pd.read_csv('olist_sellers_dataset.csv')
category    = pd.read_csv('product_category_name_translation.csv')

datasets = {
    'orders': orders, 'customers': customers, 'order_items': order_items,
    'products': products, 'payments': payments, 'reviews': reviews,
    'sellers': sellers, 'category': category
}

for name, frame in datasets.items():
    print(f'{name:15s}  rows={frame.shape[0]:>7,}  cols={frame.shape[1]}')

## 3. Data Quality Check

Before any analysis we check for **missing values** across all tables. Ignoring nulls can silently skew aggregations.

In [ ]:
print('=== Missing Value Summary ===\n')
for name, frame in datasets.items():
    nulls = frame.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'--- {name} ---')
        print(nulls.to_string())
        print()

print('Check complete. Nulls limited to known optional fields (delivery dates, category names).')

**Findings:**
- `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`: nulls exist for orders that were cancelled or not yet delivered — expected and acceptable.
- `product_category_name`: ~600 products have no category label; we handle these with a `left` join so they appear as `NaN` rather than being dropped.

## 4. Feature Engineering

### 4.1 Parse timestamps & compute `delivery_days`

In [ ]:
# Convert to datetime — do this BEFORE merging to keep the orders table clean
orders['order_purchase_timestamp']      = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

# delivery_days = actual days from purchase to delivery at customer
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

print(f"Delivery days — mean: {orders['delivery_days'].mean():.1f} | "
      f"median: {orders['delivery_days'].median():.1f} | "
      f"max: {orders['delivery_days'].max():.0f}")

### 4.2 Master merge

All 8 tables are joined into a single flat dataframe on their natural keys. We use `inner` joins throughout (except for the category translation which uses `left` to retain products with missing category names).

In [ ]:
df = (orders
      .merge(customers,   on='customer_id')
      .merge(order_items, on='order_id')
      .merge(products,    on='product_id')
      .merge(payments,    on='order_id')
      .merge(reviews,     on='order_id')
      .merge(sellers,     on='seller_id')
      .merge(category,    on='product_category_name', how='left')
)

# Derived time columns on the merged frame
df['month']  = df['order_purchase_timestamp'].dt.to_period('M')
df['year']   = df['order_purchase_timestamp'].dt.year

# Delivery bucket for segment analysis
df['delivery_bucket'] = pd.cut(
    df['delivery_days'],
    bins=[0, 3, 7, 100],
    labels=['Fast (≤3d)', 'Moderate (4–7d)', 'Delayed (>7d)'],
    include_lowest=True
)

print(f'Master dataframe shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 5. Revenue Analysis

### 5.1 Total revenue

In [ ]:
total_revenue = df['payment_value'].sum()
print(f'Total Revenue (all orders): R$ {total_revenue:,.2f}')

### 5.2 Revenue by product category (Top 10)

In [ ]:
category_revenue = (
    df.groupby('product_category_name_english')['payment_value']
      .sum()
      .sort_values(ascending=False)
)

top10 = category_revenue.head(10)
top10_share = top10.sum() / category_revenue.sum()

print(f'Top-10 categories account for {top10_share:.1%} of total revenue.\n')
print(top10.to_string())

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 5))
top10.sort_values().plot(kind='barh', color=sns.color_palette('Blues_r', 10), ax=ax)
ax.set_title('Top 10 Product Categories by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Payment Value (R$)')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

### 5.3 Monthly revenue trend

In [ ]:
monthly_rev = df.groupby('month')['payment_value'].sum()

fig, ax = plt.subplots(figsize=(13, 5))
monthly_rev.plot(ax=ax, color='steelblue', linewidth=2, marker='o', markersize=4)
ax.set_title('Monthly Revenue Trend (2016–2018)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}K'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Note: Data ends mid-2018, so the apparent drop in recent months reflects incomplete data, not a real decline.')

## 6. Delivery Performance

### 6.1 Overall delivery stats

In [ ]:
desc = df['delivery_days'].describe()
late_count = df[df['delivery_days'] > 5].shape[0]
late_pct   = late_count / df['delivery_days'].notna().sum()

print('Delivery Days Summary:')
print(desc.to_string())
print(f'\nOrders taking more than 5 days: {late_count:,} ({late_pct:.1%} of delivered orders)')

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
df['delivery_days'].dropna().clip(upper=60).hist(bins=60, color='steelblue', edgecolor='white', ax=ax)
ax.axvline(desc['mean'], color='red',    linestyle='--', label=f"Mean {desc['mean']:.1f}d")
ax.axvline(desc['50%'],  color='orange', linestyle='--', label=f"Median {desc['50%']:.0f}d")
ax.set_title('Distribution of Delivery Days (clipped at 60d)', fontsize=13, fontweight='bold')
ax.set_xlabel('Days')
ax.set_ylabel('Order count')
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Delivery speed vs. customer review score

We bin deliveries into three speed tiers and compare their average review scores to test whether faster delivery correlates with higher satisfaction.

In [ ]:
bucket_scores = (
    df.groupby('delivery_bucket', observed=True)['review_score']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'avg_review', 'count': 'order_count'})
)

print(bucket_scores.round(3).to_string())

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bucket_scores['avg_review'].plot(kind='bar', color=colors, ax=ax, width=0.5)
ax.set_ylim(3.5, 5)
ax.set_title('Avg Review Score by Delivery Speed', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Average Review Score (1–5)')
ax.set_xticklabels(bucket_scores.index, rotation=15)
for i, v in enumerate(bucket_scores['avg_review']):
    ax.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInsight: Faster delivery is clearly associated with higher satisfaction.')
print('Delayed orders (>7d) score ~0.45 points lower than fast deliveries.')

### 6.3 Review score vs. average delivery days

In [ ]:
score_vs_days = (
    df.groupby('review_score')['delivery_days']
      .mean()
      .reset_index()
)

print(score_vs_days.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(score_vs_days['review_score'], score_vs_days['delivery_days'],
        marker='o', color='steelblue', linewidth=2)
ax.set_title('Avg Delivery Days per Review Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Review Score')
ax.set_ylabel('Avg Delivery Days')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

print('\nInsight: 1-star reviews correlate with nearly double the delivery time of 5-star reviews.')

## 7. Geographic Analysis

### 7.1 Cities with slowest average delivery

In [ ]:
# Filter to cities with at least 5 orders to avoid one-off outliers skewing results
city_counts = df.groupby('customer_city')['order_id'].count()
valid_cities = city_counts[city_counts >= 5].index

city_delay = (
    df[df['customer_city'].isin(valid_cities)]
      .groupby('customer_city')['delivery_days']
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

print('Top 10 cities with slowest delivery (min 5 orders):')
print(city_delay.round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
city_delay.sort_values().plot(kind='barh', color='salmon', ax=ax)
ax.set_title('Top 10 Cities with Longest Average Delivery (≥5 orders)', fontsize=13, fontweight='bold')
ax.set_xlabel('Avg Delivery Days')
plt.tight_layout()
plt.show()

print('\nNote: Without the min-order filter, extreme values (e.g. 148 days) come from cities with a single order — statistically unreliable.')

## 8. Customer Analysis

### 8.1 Repeat purchase rate

The Olist dataset uses `customer_id` (not `customer_unique_id`), which is scoped per order. We use `customer_unique_id` for a more accurate repeat-buyer analysis.

In [ ]:
# Merge unique customer ID back in
uid_map = customers[['customer_id', 'customer_unique_id']].drop_duplicates()
df = df.merge(uid_map, on='customer_id', how='left', suffixes=('', '_lookup'))

unique_orders = df.groupby('customer_unique_id')['order_id'].nunique()
repeat_customers = unique_orders[unique_orders > 1]
repeat_pct = len(repeat_customers) / len(unique_orders)

repeat_revenue = df[df['customer_unique_id'].isin(repeat_customers.index)]['payment_value'].sum()
repeat_rev_pct  = repeat_revenue / total_revenue

print(f'Total unique customers : {len(unique_orders):,}')
print(f'Repeat buyers (>1 order): {len(repeat_customers):,}  ({repeat_pct:.2%} of customers)')
print(f'Revenue from repeat buyers: R$ {repeat_revenue:,.2f}  ({repeat_rev_pct:.2%} of total revenue)')
print()
print('Insight: Repeat purchase rate is very low — Olist has a significant customer retention opportunity.')

### 8.2 Top 10 customers by revenue

In [ ]:
top_customers = (
    df.groupby('customer_unique_id')['payment_value']
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print('Top 10 customers by total spend:')
print(top_customers.apply(lambda x: f'R$ {x:,.2f}').to_string())

fig, ax = plt.subplots(figsize=(10, 4))
top_customers.sort_values().plot(kind='barh', color='mediumpurple', ax=ax)
ax.set_title('Top 10 Customers by Total Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Spend (R$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}K'))
plt.tight_layout()
plt.show()

## 9. Payment Method Analysis

In [ ]:
payment_summary = (
    df.groupby('payment_type')['payment_value']
      .agg(['sum', 'count'])
      .rename(columns={'sum': 'total_revenue', 'count': 'transactions'})
      .sort_values('total_revenue', ascending=False)
)
payment_summary['revenue_share'] = payment_summary['total_revenue'] / payment_summary['total_revenue'].sum()

print(payment_summary.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Revenue pie
payment_summary['total_revenue'].plot(
    kind='pie', autopct='%1.1f%%', ax=axes[0],
    colors=sns.color_palette('pastel'), startangle=90
)
axes[0].set_title('Revenue Share by Payment Method', fontsize=12, fontweight='bold')
axes[0].set_ylabel('')

# Transaction count bar
payment_summary['transactions'].plot(kind='bar', color=sns.color_palette('Set2'), ax=axes[1])
axes[1].set_title('Transaction Count by Payment Method', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(payment_summary.index, rotation=20)
axes[1].set_ylabel('Number of Transactions')

plt.tight_layout()
plt.show()

print('\nInsight: Credit card dominates at ~76% of revenue. Boleto (bank slip) is a notable second — typical for Brazil.')

## 10. Seller Performance

### 10.1 Fastest sellers

In [ ]:
# Require at least 20 orders per seller for a meaningful average
seller_order_counts = df.groupby('seller_id')['order_id'].count()
active_sellers = seller_order_counts[seller_order_counts >= 20].index

seller_perf = (
    df[df['seller_id'].isin(active_sellers)]
      .groupby('seller_id')
      .agg(
          avg_delivery_days=('delivery_days', 'mean'),
          avg_review=('review_score', 'mean'),
          total_revenue=('payment_value', 'sum'),
          order_count=('order_id', 'count')
      )
      .sort_values('avg_delivery_days')
)

print('Top 10 fastest sellers (min 20 orders):')
print(seller_perf.head(10).round(2).to_string())

# Scatter: delivery speed vs review score
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(
    seller_perf['avg_delivery_days'],
    seller_perf['avg_review'],
    c=seller_perf['total_revenue'],
    cmap='viridis', alpha=0.6, s=40
)
plt.colorbar(sc, ax=ax, label='Total Revenue (R$)')
ax.set_title('Seller: Avg Delivery Days vs Avg Review Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Avg Delivery Days')
ax.set_ylabel('Avg Review Score')
plt.tight_layout()
plt.show()

print('\nInsight: Sellers with faster delivery generally earn higher review scores.')

### 10.2 Revenue vs. order price segment

In [ ]:
price_review = (
    df.groupby(pd.cut(df['payment_value'], bins=5), observed=True)['review_score']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'avg_review', 'count': 'orders'})
)

print('Review score by order value segment:')
print(price_review.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
price_review['avg_review'].plot(kind='bar', color='teal', ax=ax)
ax.set_title('Avg Review Score by Order Value Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Payment Value Range (R$)')
ax.set_ylabel('Avg Review Score')
ax.set_xticklabels([str(x) for x in price_review.index], rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 11. Summary of Key Findings

| # | Finding | Value |
|---|---------|-------|
| 1 | **Total platform revenue** | ~R$20.2M |
| 2 | **Top category** | Bed, Bath & Table (~R$1.7M) |
| 3 | **Top-10 categories share** | ~66% of all revenue |
| 4 | **Avg delivery time** | ~12 days |
| 5 | **% orders taking >5 days** | ~92% |
| 6 | **Fast delivery avg review** | 4.39 / 5 |
| 7 | **Delayed delivery avg review** | 3.95 / 5 |
| 8 | **Dominant payment method** | Credit card (~76% revenue) |
| 9 | **Repeat customer rate** | Very low (<5%) — retention gap |

---

### Recommendations
1. **Invest in last-mile logistics** — delivery speed is the single biggest lever on customer satisfaction.
2. **Focus marketing on repeat retention** — the platform is heavily acquisition-driven; even small improvements in repeat rate would significantly increase LTV.
3. **Prioritise top categories** — bed/bath, health/beauty, and electronics together make up the bulk of revenue; seller recruitment and promotions should focus here first.
4. **Flag high-delay cities** — cities with consistent 30+ day delivery times may benefit from regional warehouse partnerships.